In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/rj.workhub/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "rj.workhub"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "88fff356ea218f9049140f5c41fb57fa3e757224"

In [2]:
import os

In [3]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject/research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject'

In [6]:
# step-1: update config.yaml
# step-2: check to update params.yaml
# step-3: check to update schema.yaml
# step-4: update entity --> create class of model evaluation with dataclass
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    all_params: dict
    target_column: str
    mlflow_uri: str

# step-5: update configuration manager --> read all yaml file from config then create class with ModelEvaluationConfig
from src.my_first_end_to_end_project.constants import *
from src.my_first_end_to_end_project.utils.common_utils import read_yaml, create_directories,save_json
class ConfigurationManager:
    def __init__(
            self,
            config_filepath= CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH,
            schema_filepath= SCHEMA_FILE_PATH
            ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir= config.root_dir,
            test_data_path=config.test_data_path,
            model_path= config.model_path,
            all_params= params,
            metric_file_name= config.metric_file_name,
            target_column= schema.name,
            mlflow_uri="https://dagshub.com/rj.workhub/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects.mlflow"
        )

        return model_evaluation_config

# step-6: Creating component --> model_evaluation.py

import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

class ModelEvaluation:
    def __init__(
            self,
            config:ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual,pred):
        rmse = np.sqrt(mean_squared_error(actual,pred))
        mae= mean_absolute_error(actual,pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis =1)
        test_y = test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            pred_value = model.predict(test_x)
            (rmse, mae, r2)= self.eval_metrics(test_y,pred_value)

            # saving metrics as local
            scores = {"rmse":rmse, "mae":mae, "r2":r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2",r2)
            mlflow.log_metric("mae", mae)

            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                # register the model
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model,"model")       


In [7]:
# step-7: execution in training pipeline
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

2026-01-09 07:26:41 - INFO - common_utils - yaml file: config/config.yaml is loaded successfully 🥳


2026-01-09 07:26:41 - INFO - common_utils - yaml file: params.yaml is loaded successfully 🥳
2026-01-09 07:26:41 - INFO - common_utils - yaml file: schema.yaml is loaded successfully 🥳
2026-01-09 07:26:41 - INFO - common_utils - Created directory Successfuly at: artifacts 🥳
2026-01-09 07:26:41 - INFO - common_utils - Created directory Successfuly at: artifacts/model_evaluation 🥳
2026-01-09 07:26:43 - INFO - common_utils - Json file saved at: artifacts/model_evaluation/metric.json 🥳


2026/01/09 07:26:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticnetModel'.
2026/01/09 07:26:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run calm-frog-947 at: https://dagshub.com/rj.workhub/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects.mlflow/#/experiments/0/runs/02c21835db5b440583b5c2f89ff3e68d
🧪 View experiment at: https://dagshub.com/rj.workhub/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects.mlflow/#/experiments/0
